# Python Bindings Development Roadmap

This notebook documents what the Rust Barter engine exposes vs. what the Python
bindings currently cover.

## Current State

| Feature Area | Python Status | Rust Status |
|---|---|---|
| Instruments & Indexing | **Complete** | Complete |
| Market Data (trades, 12 exchanges) | **Complete** | Complete |
| Market Data (L1 books) | **Complete** (Binance, Kraken, Bybit) | Complete |
| Market Data (L2 books) | **Complete** (`build_order_book_manager`) | Complete |
| Trading Statistics | **Complete** | Complete |
| Engine / Backtesting | **Complete** | Full lifecycle |
| Custom Strategies | **Complete** (single + multi-strategy) | 4 traits, composition |
| Custom Risk Management | **Complete** | Full `RiskManager` trait |
| Order Management | **Complete** | Full order types |
| Position Tracking | **Complete** (per-strategy via `PyInstrumentData`) | `PositionManager` |
| Event Callbacks | **Complete** (`on_fill`, `on_position_closed`) | Via audit stream |
| Account Events | **Complete** (`TradeFill`, `AccountEvent`, `PositionExited`) | Full types |
| Live Trading | **Blocked** (no upstream execution client) | `ExecutionClient` trait |

---
## Phase 1: Strategy Callbacks — DONE

**Implemented in `barter-python` v0.2.0.**

### What was built

- `EngineState` snapshot type (`barter-python/src/state.rs`) exposing:
  - `state.price(instrument_index)` — current market price
  - `state.position(instrument_index)` — position with side, qty, entry price, PnL (realised + unrealised), trade count, timestamps
  - `state.instruments()` / `state.balances()` — full state dicts
  - `state.open_orders(instrument_index)` — active order count
- `OrderRequestOpen` and `OrderRequestCancel` types (`barter-python/src/order.rs`)
- `AlgoStrategy` trait delegated to Python callable via GIL acquisition per tick
- `run_backtest(config, data, strategy=my_fn)` parameter added

### Usage

```python
from barter import run_backtest, OrderRequestOpen

def my_strategy(state):
    price = state.price(0)
    if price and price < 100000:
        return [OrderRequestOpen(0, 0, "buy", price, 0.001)]
    return []

summary = await run_backtest(config, data, strategy=my_strategy)
```

---
## Phase 2: Risk Management Callbacks — DONE

**Implemented in `barter-python` v0.2.0.**

### What was built

- `RiskManager` trait delegated to Python callable (`barter-python/src/risk.rs`)
- Callback signature: `fn(state: EngineState, orders: List[OrderRequestOpen]) -> List[OrderRequestOpen]`
- Return only approved orders; omitted orders are rejected
- Default (no callback): approve all orders
- Python exceptions caught and logged, falls back to approve-all

### Usage

```python
def my_risk(state, orders):
    return [o for o in orders if o.price * o.quantity <= 100.0]

summary = await run_backtest(config, data, strategy=my_strategy, risk=my_risk)
```

---
## Phase 3: Position & Account Event Exposure — DONE

### What was built

- Enriched `EngineState` snapshot with full position detail
- `on_fill` callback via custom `PyGlobalData` (`barter-python/src/global.rs`)
- `on_position_closed` callback via custom `PyInstrumentData` (`barter-python/src/instrument_data.rs`)
- `TradeFill`, `AccountEvent`, `PositionExited` Python types (`barter-python/src/account.rs`)
- Per-strategy position tracking with `StrategyId`-based trade routing

### Usage

```python
def on_fill(trade):
    print(f"Fill: {trade.side} {trade.quantity} @ {trade.price}")

def on_position_closed(pos):
    print(f"Closed: PnL={pos.pnl_realised}")

summary = await run_backtest(config, data, strategy=fn,
    on_fill=on_fill, on_position_closed=on_position_closed)
```

---
## Phase 4: Extended Market Data — DONE

### What was built

- L1 order book subscriptions (`data_kind="order_book_l1"`) on Binance, Kraken, Bybit
- L2 order book depth: `build_order_book_manager()` with `OrderBook` and `OrderBookManager`
- 12 exchanges: Binance Spot/Futures, OKX, Coinbase, Kraken, Bitfinex, Bitmex,
  Bybit Spot/Perpetuals, Gateio Spot/Perpetuals/Futures
- Multi-strategy composition: `strategies={"name": fn, ...}`

### Usage

```python
# L2 order books
mgr = build_order_book_manager([Subscription("binance_spot", "btc", "usdt")])
book = mgr.get("btc_usdt_spot")
print(book.best_bid, book.best_ask, book.spread)

# Multi-strategy
summary = await run_backtest(config, data,
    strategies={"momentum": fn1, "mean_rev": fn2})
```

---
## Phase 5: Live Trading — BLOCKED

**Blocked on upstream**: `barter-execution/src/client/binance/mod.rs` is empty.
No real exchange execution client exists yet in the upstream crate.

### What's ready (all infrastructure)
- Custom `PyGlobalData` and `PyInstrumentData` type system
- Strategy, risk, on_fill, on_position_closed callbacks
- Multi-strategy composition with per-strategy position tracking

### What's blocked on upstream
- Real `ExecutionClient` implementation for any exchange
- `ExecutionConfig::Live` variant for real exchange credentials
- Exchange API key management

### Proposed API (when unblocked)

```python
system = await run_live(
    config_json=config,
    subscriptions=[Subscription("binance_spot", "btc", "usdt")],
    strategy=my_strategy,
    risk=my_risk,
    on_fill=on_fill,
    trading_enabled=False,
)
system.enable_trading()
summary = await system.shutdown()
```

---
## Implementation Status

```
                    DONE                         BLOCKED
    ┌──────────────────────────────┬──────────────────────────┐
    │ Phase 1: Strategy Callbacks  │ Phase 5: Live Trading    │
    │ Phase 2: Risk Callbacks      │   - Needs upstream       │
    │ Phase 3: Event Callbacks     │     ExecutionClient      │
    │   - on_fill                  │   - run_live()           │
    │   - on_position_closed       │   - OnDisconnect         │
    │   - TradeFill, PositionExited│   - OnTradingDisabled    │
    │   - AccountEvent types       │   - PySystem wrapper     │
    │ Phase 4: Market Data         │                          │
    │   - 12 exchanges             │                          │
    │   - L1 + L2 order books      │                          │
    │   - Multi-strategy           │                          │
    │   - Per-strategy positions   │                          │
    └──────────────────────────────┴──────────────────────────┘
```

---
## Technical Notes

### GIL Considerations

Calling Python callbacks from the Rust engine requires acquiring the GIL on
each engine tick. For backtesting this is acceptable (the engine is
single-threaded and I/O-free). For live trading, consider:

- Releasing the GIL during Rust-only processing (`py.allow_threads()`)
- Batching multiple events before calling back into Python
- Using `tokio::spawn_blocking()` for the Python callback to avoid blocking
  the async runtime

### State Snapshot Pattern (Implemented)

The strategy and risk callbacks use the **snapshot** pattern: engine state is
copied into Python-owned objects before each callback, avoiding Rust borrow
lifetime issues across the FFI boundary.

```
Engine tick → copy state to PyEngineStateSnapshot → acquire GIL → call Python
           → collect returned orders → release GIL → feed orders to risk manager
```

### Error Handling

Python exceptions in callbacks are caught gracefully:
- **Strategy callback error**: logged to stderr, returns no orders (noop tick)
- **Risk callback error**: logged to stderr, approves all orders (fail-open)

This ensures the engine never panics due to Python errors.